In [226]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import html
import contractions
import nltk
import os

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import euclidean_distances

import joblib
import random

import difflib
import re

In [138]:
# text preprocessing objects
svd_model = joblib.load('../../pickle-object/svd.pkl')
tfidf_vectorizer = joblib.load('../../pickle-object/tfidf.pkl')
final_n_svs = joblib.load('../../pickle-object/final_n_svs.pkl')

# model prediction objects
X_columns = joblib.load('../../pickle-object/X_columns.pkl')
model = joblib.load('../../pickle-object/lasso.pkl')

# for clustering
ref_df = pd.read_csv('../../pickle-object/clusters.csv', index_col='Cluster_ID')

In [139]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_nltk_alpha_only(text):
    text = html.unescape(str(text))
    text = contractions.fix(text)

    tokens = word_tokenize(text.lower())

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)

In [140]:
def description_to_svd_vector(description):
    processed_text = preprocess_nltk_alpha_only(description)

    tfidf_vector = tfidf_vectorizer.transform([processed_text])

    svd_vector = svd_model.transform(tfidf_vector)[:, :final_n_svs]

    svd_vector_df = pd.DataFrame(
        svd_vector,
        columns=[f"SV_{i+1}" for i in range(final_n_svs)]
    )

    return svd_vector_df

In [141]:
svd_cols = [col for col in ref_df.columns if col.startswith('SV_')]
centroids_matrix = ref_df[svd_cols].values
competition_scores = ref_df['Competition_Score'].to_dict()

scaler = Normalizer(norm='l2')

def clustering(svd_vector_df):
    """Takes the SVD dataframe and instantly finds its market position."""
    
    new_scaled = scaler.transform(svd_vector_df.values)
    
    distances = euclidean_distances(new_scaled, centroids_matrix)[0]
    
    cluster = ref_df.index[np.argmin(distances)]
    competition_score = competition_scores[cluster]
    
    return cluster, competition_score

In [142]:
def predict_output(vec, desc_only, features=None):
    if desc_only:
        return model.predict(vec)
    data = pd.DataFrame(data=vec, columns=X_columns).fillna(0)
    for feature in features:
        data.at[0, feature] = 1
    
    return model.predict(data)

In [143]:
def run_pipeline(description=None, bin=False, bin_method=round, desc_only=False, mechs_and_cats=[]):
    if not desc_only:
        if not mechs_and_cats:
            return
    
    if description == None:
        description = input('Input game description: ')
        
    svd_df = description_to_svd_vector(description)

    predicted_score = predict_output(svd_df, desc_only, mechs_and_cats)[0]
    if bin:
        predicted_score = bin_method(predicted_score)
    cluster, competition_score = clustering(svd_df)

    print(f"Predicted Score: {predicted_score:.2f}")
    print(f"Cluster: {cluster}, Competition Score: {competition_score}")

In [231]:
def find_closest_columns(user_input, X_columns):
    if user_input in X_columns:
        return user_input

    try:
        escaped_input = re.escape(user_input)
        pattern = re.compile(escaped_input, re.IGNORECASE)
        regex_matches = [col for col in X_columns if pattern.search(col)]
    except re.error:
        regex_matches = []

    regex_matches = sorted(
        regex_matches, 
        key=lambda x: difflib.SequenceMatcher(None, user_input.lower(), x.lower()).ratio(), 
        reverse=True
    )

    if regex_matches:
        return regex_matches[:10]
    else:
        return difflib.get_close_matches(user_input, X_columns, n=1, cutoff=0.3)[0]

In [234]:
description = """
A competitive deck-building mystery game of deduction, deception, and hidden agendas.
Investigate a dynamic case by gathering clues, questioning suspects, and manipulating the evidence,
all while building a unique deck of investigative tactics, bluffs, and dirty tricks. Play solo,
cooperatively, or against each other in teams as you uncover the truth, or bury it deep enough
to escape suspicion. Inspired by the tension of classic murder mysteries and the mind games of
social deduction, every case becomes a battle of logic, memory, and lies.
"""

random_items = random.sample(list(X_columns)[1804:], 15)

sample_features = ["Bluffing",
                "Crime (Spy / Espionage)",
                "Semi-Cooperative Game",
                "Traitor Game",
                "Team-Based Game",
                "Role Playing"]

run_pipeline(description, bin=False, bin_method=round, desc_only=False, mechs_and_cats=sample_features)

Predicted Score: 8.95
Cluster: 6, Competition Score: 1.0


In [242]:
description = """
This playing system serves as an introduction to the Aethelgard universe. As a role playing rpg experience, each character functions as a traveller on a white road. Instead of miniatures, the system utilizes a monopoly of plastic rings and numbered pawns.  

A summary of the playing is provided in a 200-page introduction document. Before playing, a user must engage in a mandatory four-hour setup role to verify the character system. On a turn, you must roll a die to see if you are allowed to move; if you fail, you lose a turn immediately.  

If you are allowed to move, you roll / spin and move your character along the white road. If you land on a user hex, you must challenge the player to your left to Rock-Paper-Scissors. The loser is eliminated from the rpg universe and must leave the room.  

There is no interesting fortune or strategy here. The playing simply continues until the system has eliminated every character through monopoly and random chance. This rpg summary features a 40-page section for character ties, ensuring the playing system is as tedious as the introduction.
"""

sample_features = ['Mythology',
 'Travel',
 'Pirates',
 'Roll / Spin and Move',
 'Player Elimination',
 'Lose a Turn',
 'Rock-Paper-Scissors',
 'Memory']

run_pipeline(description, bin=False, bin_method=round, desc_only=False, mechs_and_cats=sample_features)

Predicted Score: 2.46
Cluster: 6, Competition Score: 1.0


In [232]:
[find_closest_columns(i, X_columns) for i in test]

['Accessory (dice, maps, screens, cards)',
 'Turn Order: Stat-Based',
 'Comedy / Satire, Science Fiction (Post Apocalypse)',
 'History (Prehistoric)',
 'Roll / Spin and Move',
 'Point to Point Movement',
 'Software (for maps, char sheets, etc)',
 'Fantasy (Modern Urban Fantasy)',
 'Fantasy, Science Fiction (Space Opera)',
 'Fantasy (Arabian Nights)']

In [244]:
f = ['Monopoly',
'Traveller',
'White',
'Roll / Spin and Move',
'Player Elimination',
'Lose a Turn',
'Rock-Paper-Scissors',
'Memory'
]
for i in [find_closest_columns(i, X_columns) for i in f]:
    print(i)


Mythology
Travel
Pirates
Roll / Spin and Move
Player Elimination
Lose a Turn
Rock-Paper-Scissors
Memory


In [243]:
for i in f:
    print(i)

Monopoly
Traveller
White
Roll / Spin and Move
Player Elimination
Lose a Turn
Rock-Paper-Scissors
Memory


In [245]:
competition_scores

{0: 0.468,
 1: 0.006,
 2: 0.0066,
 3: 0.006,
 4: 0.0064,
 5: 0.0943,
 6: 1.0,
 7: 0.0095,
 8: 0.0029,
 9: 0.0023,
 10: 0.0015,
 11: 0.0012,
 12: 0.005,
 13: 0.0015,
 14: 0.0002,
 15: 0.0031,
 16: 0.0477,
 17: 0.0122,
 18: 0.0023,
 19: 0.0282,
 20: 0.0091,
 21: 0.0133,
 22: 0.0012,
 23: 0.0276,
 24: 0.0017,
 25: 0.0085,
 26: 0.0006,
 27: 0.0112,
 28: 0.001,
 29: 0.0039,
 30: 0.0427,
 31: 0.028,
 32: 0.0}